# VLA Finetuning Eval v2

This notebook loads a finetuned VLA checkpoint, builds the `inst_dataloader_v2` evaluation loader, and reports instruction / subgoal metrics for the new `Instruction: ... Subgoal: ...` answer format.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
import re
import sys
from dataclasses import fields
from pathlib import Path
from textwrap import shorten

import torch

sys.path.insert(0, "/zfsauton2/home/mineuih/waymax_rs")

from data.inst_dataloader_v2 import build_inst_dataloader
from data.types import PreprocessConfig
from data.utils import resolve_cache_paths, split_cache_paths
from train_vla.configs.vla_finetuning_config import VLAFinetuningConfig
from train_vla.utils.utils import (
    build_vla_model,
    move_features_to_device,
    resolve_device,
    resolve_dtype,
    tokenize_text_batch,
)

CHECKPOINT_OR_RUN_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260603_154504/checkpoints/step_00100000.pt")
CACHE_DIR_OVERRIDE: str | None = None
INSTRUCTION_DIR_OVERRIDE: str | None = None
FILE_INDICES_OVERRIDE: list[int] | None = None
VALIDATION_FRACTION_OVERRIDE: float | None = None
BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = False
MAX_SAMPLES = 160
MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 64
SHOW_N_SAMPLES = 50


def resolve_run_dir_and_checkpoint(path: Path) -> tuple[Path, Path]:
    path = path.expanduser().resolve()
    if path.is_file():
        if path.parent.name == "checkpoints":
            return path.parent.parent, path
        return path.parent, path
    if path.is_dir():
        if path.name == "checkpoints":
            checkpoint_files = sorted(path.glob("step_*.pt"))
            if not checkpoint_files:
                raise FileNotFoundError(f"No checkpoints found in {path}")
            return path.parent, checkpoint_files[-1]
        checkpoint_dir = path / "checkpoints"
        if checkpoint_dir.exists():
            checkpoint_files = sorted(checkpoint_dir.glob("step_*.pt"))
            if not checkpoint_files:
                raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
            return path, checkpoint_files[-1]
    raise FileNotFoundError(f"Could not resolve run directory from: {path}")


def load_run_config(run_dir: Path) -> dict:
    config_path = run_dir / "training_config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"training_config.json not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def filter_finetuning_config(config_data: dict) -> dict:
    valid_keys = {field.name for field in fields(VLAFinetuningConfig)}
    return {key: value for key, value in config_data.items() if key in valid_keys}


def load_preprocess_cfg(config_data: dict) -> PreprocessConfig:
    preprocess_cfg = config_data.get("preprocess_cfg", {})
    if isinstance(preprocess_cfg, PreprocessConfig):
        return preprocess_cfg
    if isinstance(preprocess_cfg, dict):
        return PreprocessConfig(**preprocess_cfg)
    return PreprocessConfig()


def strip_module_prefix(state_dict: dict) -> dict:
    if not any(key.startswith("module.") for key in state_dict):
        return state_dict
    return {
        key[len("module.") :] if key.startswith("module.") else key: value
        for key, value in state_dict.items()
    }


def load_model_from_checkpoint(checkpoint_path: Path) -> tuple[torch.nn.Module, dict, Path, Path, torch.device, torch.dtype]:
    run_dir, resolved_checkpoint = resolve_run_dir_and_checkpoint(checkpoint_path)
    run_config = load_run_config(run_dir)
    model_config = filter_finetuning_config(run_config)
    model = build_vla_model(model_config)

    checkpoint = torch.load(resolved_checkpoint, map_location="cpu")
    state_dict = strip_module_prefix(checkpoint.get("model_state_dict", checkpoint))
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"run_dir: {run_dir}")
    print(f"checkpoint: {resolved_checkpoint}")
    print(f"missing keys: {len(missing)}")
    print(f"unexpected keys: {len(unexpected)}")
    if missing:
        print("  missing:", missing[:10])
    if unexpected:
        print("  unexpected:", unexpected[:10])

    device = resolve_device()
    dtype = resolve_dtype(str(run_config.get("dtype", "bf16"))) if device.type == "cuda" else torch.float32
    model = model.to(device=device, dtype=dtype)
    model.eval()
    return model, run_config, run_dir, resolved_checkpoint, device, dtype


DirectionBucket = str
SpeedBucket = str


DIRECTION_BUCKETS: tuple[str, ...] = (
    "stop",
    "straight",
    "slight_left",
    "slight_right",
    "turn_left",
    "turn_right",
    "u_turn",
    "other",
)

SPEED_BUCKETS: tuple[str, ...] = (
    "accelerating",
    "slowing_down",
    "constant_or_unspecified",
)


def normalize_instruction(text: str) -> str:
    normalized = str(text).lower().strip()
    normalized = re.sub(r"\s+", " ", normalized)
    normalized = normalized.rstrip(".,;:")
    return normalized


def direction_bucket(text: str) -> DirectionBucket:
    normalized = normalize_instruction(text)
    if not normalized:
        return "other"
    if normalized == "stop" or normalized.startswith("stop "):
        return "stop"
    if "u-turn" in normalized or "u turn" in normalized or "make a u-turn" in normalized:
        return "u_turn"
    if "slightly left" in normalized or "slight left" in normalized:
        return "slight_left"
    if "slightly right" in normalized or "slight right" in normalized:
        return "slight_right"
    if "turn left" in normalized or "to turn left" in normalized:
        return "turn_left"
    if "turn right" in normalized or "to turn right" in normalized:
        return "turn_right"
    if "changing lane to the left" in normalized or "change lane to the left" in normalized:
        return "slight_left"
    if "changing lane to the right" in normalized or "change lane to the right" in normalized:
        return "slight_right"
    if normalized.startswith("go left") or " go left" in normalized:
        return "turn_left"
    if normalized.startswith("go right") or " go right" in normalized:
        return "turn_right"
    if "go straight" in normalized or "following current lane" in normalized:
        return "straight"
    return "other"


def speed_bucket(text: str) -> SpeedBucket:
    normalized = normalize_instruction(text)
    if "accelerat" in normalized:
        return "accelerating"
    if "slowing down" in normalized or "slow down" in normalized or "slowing" in normalized:
        return "slowing_down"
    return "constant_or_unspecified"


def direction_bucket_match(pred: str, target: str) -> bool:
    return direction_bucket(pred) == direction_bucket(target)


def speed_bucket_match(pred: str, target: str) -> bool:
    return speed_bucket(pred) == speed_bucket(target)


def semantic_bucket_match(pred: str, target: str) -> bool:
    return direction_bucket_match(pred, target) and speed_bucket_match(pred, target)


def split_instruction_subgoal_text(text: str, model: torch.nn.Module | None = None) -> tuple[str, str]:
    if model is not None:
        splitter = getattr(model, "split_instruction_subgoal_text", None)
        if callable(splitter):
            return splitter(text)

    instruction_marker = "Instruction:"
    subgoal_marker = "Subgoal:"
    instruction_start = text.find(instruction_marker)
    subgoal_start = text.find(subgoal_marker)

    if instruction_start == -1 and subgoal_start == -1:
        return text.strip(), ""
    if instruction_start == -1:
        instruction_text = text[:subgoal_start].strip()
    else:
        instruction_begin = instruction_start + len(instruction_marker)
        if subgoal_start == -1:
            instruction_text = text[instruction_begin:].strip()
        else:
            instruction_text = text[instruction_begin:subgoal_start].strip()
    if subgoal_start == -1:
        subgoal_text = ""
    else:
        subgoal_begin = subgoal_start + len(subgoal_marker)
        subgoal_text = text[subgoal_begin:].strip()
    return instruction_text, subgoal_text


def parse_subgoal_point(text: str) -> tuple[float, float] | None:
    cleaned = str(text).strip().strip("()").replace(" ", "")
    if not cleaned:
        return None
    if "," not in cleaned:
        return None
    x_str, y_str = cleaned.split(",", 1)
    try:
        return float(x_str), float(y_str)
    except ValueError:
        return None


def generate_instruction_and_subgoal(
    model: torch.nn.Module,
    features: dict[str, torch.Tensor],
    prompt_ids: torch.Tensor,
    prompt_mask: torch.Tensor,
    max_new_tokens: int,
) -> tuple[list[str], list[str]]:
    generator = getattr(model, "generate_inst_subgoal_predictions", None)
    if callable(generator):
        return generator(
            input_features=features,
            prompt_ids=prompt_ids,
            prompt_mask=prompt_mask,
            max_new_tokens=max_new_tokens,
        )

    predictions = model.generate_predictions(
        input_features=features,
        prompt_ids=prompt_ids,
        prompt_mask=prompt_mask,
        max_new_tokens=max_new_tokens,
    )
    instruction_texts: list[str] = []
    subgoal_texts: list[str] = []
    for text in predictions:
        instruction_text, subgoal_text = split_instruction_subgoal_text(text, model=model)
        instruction_texts.append(instruction_text)
        subgoal_texts.append(subgoal_text)
    return instruction_texts, subgoal_texts


model, run_config, run_dir, checkpoint_path, device, dtype = load_model_from_checkpoint(CHECKPOINT_OR_RUN_DIR)
preprocess_cfg = load_preprocess_cfg(run_config)
cache_dir = CACHE_DIR_OVERRIDE or str(run_config["cache_dir"])
instruction_dir = INSTRUCTION_DIR_OVERRIDE or str(run_config["instruction_dir"])
file_indices = FILE_INDICES_OVERRIDE if FILE_INDICES_OVERRIDE is not None else run_config.get("file_indices")
validation_fraction = (
    VALIDATION_FRACTION_OVERRIDE
    if VALIDATION_FRACTION_OVERRIDE is not None
    else float(run_config.get("validation_fraction", 0.2))
)
cache_paths = resolve_cache_paths(cache_dir, file_indices)
train_cache_paths, val_cache_paths = split_cache_paths(cache_paths, validation_fraction)
eval_cache_paths = val_cache_paths or train_cache_paths or cache_paths

loader = build_inst_dataloader(
    cache_dir,
    preprocess_cfg=preprocess_cfg,
    file_indices=None,
    instruction_dir=instruction_dir,
    batch_size=BATCH_SIZE,
    shuffle_seed=0,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    cache_paths=eval_cache_paths,
)

print(f"device: {device}")
print(f"dtype: {dtype}")
print(f"run_dir: {run_dir}")
print(f"checkpoint: {checkpoint_path}")
print(f"cache_dir: {cache_dir}")
print(f"instruction_dir: {instruction_dir}")
print(f"validation_fraction: {validation_fraction}")
print(f"num eval shards: {len(eval_cache_paths)}")

instruction_match_scores: list[float] = []
direction_bucket_scores: list[float] = []
speed_bucket_scores: list[float] = []
semantic_bucket_scores: list[float] = []
subgoal_l2_distances: list[float] = []
subgoal_format_scores: list[float] = []
preview_rows: list[dict[str, str]] = []
seen = 0

with torch.inference_mode():
    for batch in loader:
        features = batch.features
        prompts = list(batch.prompts)
        answers = list(batch.answers)

        if not prompts:
            continue

        remaining = MAX_SAMPLES - seen
        if remaining <= 0:
            break
        if len(prompts) > remaining:
            prompts = prompts[:remaining]
            answers = answers[:remaining]
            features = {key: value[:remaining] for key, value in features.items()}

        prompt_ids, prompt_mask, _, _ = tokenize_text_batch(
            model.tokenizer,
            prompts,
            answers,
            add_eos=bool(run_config.get("add_eos", False)),
            device=device,
            max_prompt_length=MAX_PROMPT_LENGTH,
            max_answer_length=MAX_ANSWER_LENGTH,
        )
        features = move_features_to_device(features, device, dtype)

        amp_enabled = device.type == "cuda"
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
            predicted_instructions, predicted_subgoals = generate_instruction_and_subgoal(
                model=model,
                features=features,
                prompt_ids=prompt_ids,
                prompt_mask=prompt_mask,
                max_new_tokens=MAX_ANSWER_LENGTH,
            )

        for prompt, answer, pred_instruction, pred_subgoal in zip(
            prompts,
            answers,
            predicted_instructions,
            predicted_subgoals,
        ):
            target_instruction, target_subgoal = split_instruction_subgoal_text(answer, model=model)
            pred_instruction_norm = normalize_instruction(pred_instruction)
            target_instruction_norm = normalize_instruction(target_instruction)

            instruction_match_scores.append(
                1.0 if pred_instruction_norm == target_instruction_norm else 0.0
            )
            direction_bucket_scores.append(
                1.0 if direction_bucket_match(pred_instruction_norm, target_instruction_norm) else 0.0
            )
            speed_bucket_scores.append(
                1.0 if speed_bucket_match(pred_instruction_norm, target_instruction_norm) else 0.0
            )
            semantic_bucket_scores.append(
                1.0 if semantic_bucket_match(pred_instruction_norm, target_instruction_norm) else 0.0
            )

            target_point = parse_subgoal_point(target_subgoal)
            pred_point = parse_subgoal_point(pred_subgoal)
            if target_point is None or pred_point is None:
                subgoal_format_scores.append(0.0)
            else:
                dx = pred_point[0] - target_point[0]
                dy = pred_point[1] - target_point[1]
                subgoal_l2_distances.append((dx * dx + dy * dy) ** 0.5)
                subgoal_format_scores.append(1.0)

            if len(preview_rows) < SHOW_N_SAMPLES:
                preview_rows.append(
                    {
                        "prompt": prompt,
                        "pred_instruction": pred_instruction,
                        "pred_subgoal": pred_subgoal,
                        "target_instruction": target_instruction,
                        "target_subgoal": target_subgoal,
                    }
                )

            seen += 1
            if seen >= MAX_SAMPLES:
                break
        if seen >= MAX_SAMPLES:
            break

print(f"samples: {seen}")
print(f"instruction exact match: {sum(instruction_match_scores) / len(instruction_match_scores):.2%}")
print(f"direction bucket accuracy: {sum(direction_bucket_scores) / len(direction_bucket_scores):.2%}")
print(f"speed bucket accuracy: {sum(speed_bucket_scores) / len(speed_bucket_scores):.2%}")
print(f"semantic bucket accuracy: {sum(semantic_bucket_scores) / len(semantic_bucket_scores):.2%}")
print(f"subgoal format accuracy: {sum(subgoal_format_scores) / len(subgoal_format_scores):.2%}")
print(
    f"subgoal L2 distance: {sum(subgoal_l2_distances) / len(subgoal_l2_distances):.4f}"
    if subgoal_l2_distances
    else "subgoal L2 distance: n/a"
)

for idx, row in enumerate(preview_rows, start=1):
    print("=" * 100)
    print(f"sample {idx}")
    print(f"prompt: {shorten(row['prompt'], width=300, placeholder='...')}")
    print(f"pred instruction: {shorten(row['pred_instruction'], width=300, placeholder='...')}")
    print(f"pred subgoal: {row['pred_subgoal']}")
    print(f"target instruction: {shorten(row['target_instruction'], width=300, placeholder='...')}")
    print(f"target subgoal: {row['target_subgoal']}")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

run_dir: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260603_154504
checkpoint: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260603_154504/checkpoints/step_00100000.pt
missing keys: 0
unexpected keys: 0
device: cuda
dtype: torch.bfloat16
run_dir: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260603_154504
checkpoint: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260603_154504/checkpoints/step_00100000.pt
cache_dir: /zfsauton/scratch/mineuih/waymax_rs/sim_state_cache_npz/
instruction_dir: /zfsauton/scratch/mineuih/waymax_rs/new_instructions/
validation_fraction: 0.001
num eval shards: 1
samples: 160
instruction exact match: 14.37%
direction bucket accuracy: 44.38%
speed bucket accuracy: 50.00%
semantic bucket accuracy: 18.12%
subgoal format accuracy: 100.00%
subgoal L2 distance: 2.0859
sample 1
prompt: Given a driving scenario, propose a driving instruction and a subgoal tha